# **Course Work 2**

---

## Notebook objectives
Deploy ML Pipeline for regression or classification task using an
appropriate dataset, data processing Python libraries and Machine Learning
model.

## Inputs
Given Data Set 14 - Image classification with CNN

Data set https://www.kaggle.com/datasets/alessiocorrado99/animals10/data

Pre-processed dataset `../datasets/.npz`

## Outputs

XYZ

## Notes and comments

XZY

## Resources 
https://github.com/Code-Institute-Org/Malaria-walkthrough-ML/blob/main/jupyter_notebooks/01%20-%20DataCollection.ipynb

---

### 1) Import Librarys

In [ ]:
%pip install numpy pandas  matplotlib seaborn scikit-learn tensorflow plotly kaggle==1.5.12 kagglehub

### Import packages

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import os
import itertools
import random
import PIL
import kagglehub
from PIL import Image

import tensorflow as tf
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Conv2D, MaxPool2D, Flatten, Dropout
from tensorflow.keras.callbacks import EarlyStopping

from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix

import joblib

#settings
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'  # Show error messages only
sns.set_style('whitegrid')

---

### 2) Load data , importing data set and translating names

Code came from kaggle when you click download

In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("alessiocorrado99/animals10")

print("Path to dataset files:", path)

Code came from kaggle under data card the reason why im using it is because "Python dict containing mapping between italian (original) and english names of animals. Since the dataset was initially posted with italian names, and a lot of interesting notebooks use hardcoded names, I preferred to add this file than changing the directory names."

In [ ]:
LABEL_MAP = {
    'cane':       'dog',
    'cavallo':    'horse',
    'elefante':   'elephant',
    'farfalla':   'butterfly',
    'gallina':    'chicken',
    'gatto':      'cat',
    'mucca':      'cow',
    'pecora':     'sheep',
    'ragno':      'spider',
    'scoiattolo': 'squirrel'
}

CODE CAME FROM HELP OF AI BECAUSE
Locate the folder that actually contains the Animals-10 class folders.
kagglehub downloads this dataset with an extra nested folder, usually
called "raw-img", so using the download root directly can fail.


In [ ]:
DOWNLOAD_PATH = kagglehub.dataset_download('alessiocorrado99/animals10')
def find_animals10_image_dir(download_path, label_map):
    """Return the directory containing the expected Animals-10 class folders."""
    expected = set(label_map.keys())

    # Check the download path itself first, then common nested locations.
    candidate_paths = [download_path, os.path.join(download_path, 'raw-img')]

    # Also search a few levels down in case Kaggle changes the archive layout.
    for root, dirs, _ in os.walk(download_path):
        if root.count(os.sep) - str(download_path).count(os.sep) > 3:
            dirs[:] = []
            continue
        candidate_paths.append(root)

    seen = set()
    for candidate in candidate_paths:
        candidate = os.path.abspath(candidate)
        if candidate in seen or not os.path.isdir(candidate):
            continue
        seen.add(candidate)
        child_dirs = set(d for d in os.listdir(candidate) if os.path.isdir(os.path.join(candidate, d)))
        if expected.issubset(child_dirs):
            return candidate

    raise FileNotFoundError(
        "Could not find the Animals-10 image folders. Expected folders like "
        f"{sorted(expected)[:3]} under {download_path!r}. Check the downloaded dataset layout."
    )

DATASET_PATH = find_animals10_image_dir(DOWNLOAD_PATH, LABEL_MAP)

CLASS_NAMES = list(LABEL_MAP.values())
N_LABELS    = len(CLASS_NAMES)
IMG_SIZE    = 64 #Resize all images to 64x64 pixels

print(f"Download path: {DOWNLOAD_PATH}")
print(f"Using image folder: {DATASET_PATH}")
print(f"Number of classes: {N_LABELS}")
print(f"Classes: {CLASS_NAMES}")
print(f"Image target size: {IMG_SIZE}x{IMG_SIZE} px")


---

### 3) Data Validation and Preparation

Checking how many images there are and that it imported correctly

In [ ]:
# Verify folder structure
folders = [f for f in LABEL_MAP.keys() if os.path.isdir(os.path.join(DATASET_PATH, f))]
missing = [f for f in LABEL_MAP.keys() if f not in folders]

if missing:
    raise FileNotFoundError(f"Missing expected class folders: {missing}")

print(f"Folders found: {folders}")

# Count images per class
image_counts = {}
for folder in folders:
    folder_path = os.path.join(DATASET_PATH, folder)
    files = [f for f in os.listdir(folder_path) if f.lower().endswith(('.jpg', '.jpeg', '.png'))]
    english_name = LABEL_MAP.get(folder, folder)
    image_counts[english_name] = len(files)
    print(f"  {english_name:12s} ({folder}): {len(files)} images")

print(f"\nTotal images: {sum(image_counts.values())}")

### Validate Images

Check that images can open and following lesson 8.2

In [ ]:
def check_images(dataset_path, label_map, sample_size=100):
    """
    Validates a random sample of images from the dataset.
    Checks:
    * File can be opened by PIL
    * Image has 3 colour channels (RGB)
    * Image is not empty / corrupt (no NaN after conversion)
    """
    invalid_count = 0
    valid_count   = 0
    sampled = []

    for folder, english in label_map.items():
        folder_path = os.path.join(dataset_path, folder)
        if not os.path.isdir(folder_path):
            print(f"WARNING: missing folder for {english}: {folder_path}")
            invalid_count += 1
            continue

        files = [f for f in os.listdir(folder_path) if f.lower().endswith(('.jpg', '.jpeg', '.png'))]
        if not files:
            print(f"WARNING: no image files found for {english}: {folder_path}")
            invalid_count += 1
            continue

        sampled.extend([
            (os.path.join(folder_path, f), english)
            for f in random.sample(files, min(sample_size, len(files)))
        ])

    for img_path, label in sampled:
        try:
            img = Image.open(img_path).convert('RGB')
            arr = np.array(img)
            if arr.ndim != 3 or arr.shape[2] != 3:
                print(f"WARNING ({label}): unexpected channels — {img_path}")
                invalid_count += 1
                continue
            if np.isnan(arr).any():
                print(f"WARNING ({label}): NaN values — {img_path}")
                invalid_count += 1
                continue
            valid_count += 1
        except Exception as e:
            print(f"ERROR ({label}): {img_path} — {e}")
            invalid_count += 1

    print(f"\nValidation complete: {valid_count} valid | {invalid_count} invalid (from {len(sampled)} sampled images)")


print("Checking Images...\n")
check_images(DATASET_PATH, LABEL_MAP, sample_size=50)


### Load and resize images into numpy array

all images are 28x28 pixels and converted into RGB

In [ ]:

def load_images(dataset_path, label_map, img_size, max_per_class=None):

    images = []
    labels = []

    for idx, (folder, english) in enumerate(label_map.items()):
        folder_path = os.path.join(dataset_path, folder)
        if not os.path.isdir(folder_path):
            raise FileNotFoundError(f"Missing folder for '{english}': {folder_path}")

        files = [f for f in os.listdir(folder_path) if f.lower().endswith(('.jpg', '.jpeg', '.png'))]
        if max_per_class is not None:
            files = random.sample(files, min(max_per_class, len(files)))

        loaded = 0
        for fname in files:
            try:
                img = Image.open(os.path.join(folder_path, fname)).convert('RGB')
                img = img.resize((img_size, img_size))
                images.append(np.array(img))
                labels.append(idx)
                loaded += 1
            except Exception as e:
                print(f"Skipped corrupt/unreadable file: {os.path.join(folder_path, fname)} ({e})")

        print(f"  Loaded {loaded} images for '{english}' ({folder})")

    if not images:
        raise ValueError("No images were loaded. Check DATASET_PATH and folder names.")

    X = np.array(images, dtype='uint8')
    y = np.array(labels, dtype='int32')
    return X, y


print("Loading images...\n")
# max_per_class limits images per class to speed up training.
# Increase or set to None to use the full dataset.
X, y = load_images(DATASET_PATH, LABEL_MAP, img_size=IMG_SIZE, max_per_class=500)

print(f"\nX shape: {X.shape}  |  y shape: {y.shape}")
print(f"Pixel value range: {X.min()} – {X.max()}")


### Split into train, validation and test sets

Following lessons 8.2 and 9.1 - 80% train and 20 test

In [ ]:
# First split train and test
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=0,
    stratify=y  # Maintain class balance
)

# Second split validation from train
X_train, X_val, y_train, y_val = train_test_split(
    X_train, y_train,
    test_size=0.2,
    random_state=0,
    stratify=y_train
)

print("* Train set:     ", X_train.shape, y_train.shape)
print("* Validation set:", X_val.shape,   y_val.shape)
print("* Test set:      ", X_test.shape,  y_test.shape)

Scaling Copied from lesson 9.1

| Before Scaling (uint8) | After Scaling (float32) |
|---|---|
| Pixel values: 0 - 255 | Pixel values: 0.0 - 1.0 |
| Integer type (uint8) | Floating-point (float32)|
| Can cause instability in training | Helps stable and faster training |


In [ ]:
X_train = X_train.astype('float32') / 255.0
X_val   = X_val.astype('float32')   / 255.0
X_test  = X_test.astype('float32')  / 255.0

print(f"X_train max after scaling: {X_train.max():.1f}")
print(f"X_train shape: {X_train.shape}  — (samples, height, width, channels)")

In [ ]:
# Convert labels to categorical (one-hot encoding)
y_train_cat = to_categorical(y_train, num_classes=N_LABELS)
y_val_cat   = to_categorical(y_val,   num_classes=N_LABELS)
y_test_cat  = to_categorical(y_test,  num_classes=N_LABELS)

print(f"y_train_cat shape: {y_train_cat.shape}")
print(f"Example one-hot vector: {y_train_cat[0]}")

### Save procesed data

Code from lesson 8.2 and 9.1

In [ ]:
os.makedirs('../datasets', exist_ok=True)

np.savez_compressed(
    '../datasets/animals10_processed.npz',
    X_train=X_train,
    X_val=X_val,
    X_test=X_test,
    y_train=y_train_cat,
    y_val=y_val_cat,
    y_test=y_test_cat
)

print("Processed dataset saved to '../datasets/animals10_processed.npz'")

---

### 4) EDA

Lable frequency

Code from lesson 9.1

In [ ]:
df_freq = pd.DataFrame(columns=['Set', 'Label', 'Frequency']) # AI helped fix this line because i kept getting errors

def count_labels(dataset, dataset_name):
    global df_freq
    unique, counts = np.unique(dataset, return_counts=True)
    for label, frequency in zip(unique, counts):
        df_freq = pd.concat([
            df_freq,
            pd.DataFrame([{'Set': dataset_name, 'Label': CLASS_NAMES[label], 'Frequency': frequency}])
        ], ignore_index=True)
        print(f"* {dataset_name} - {CLASS_NAMES[label]}: {frequency} images")


count_labels(y_train, 'Train')
print()
count_labels(y_val,   'Validation')
print()
count_labels(y_test,  'Test')

In [ ]:
# Visualize the label distribution and save image
sns.set_style("whitegrid")
plt.figure(figsize=(10, 6))
sns.barplot(data=df_freq, x='Set', y='Frequency', hue='Label')
plt.xticks(rotation=45)
plt.title("Label Frequency Distribution in Train, Validation, and Test Sets")
plt.show()

Sample images

Code from kaggle

In [ ]:
sns.set_style('white')
fig, axes = plt.subplots(2, 5, figsize=(15, 6))
axes = axes.flatten()

for class_idx in range(N_LABELS):
    # Find first occurrence of this class in train set
    match_indices = np.where(y_train == class_idx)[0]
    if len(match_indices) > 0:
        sample_img = X_train[match_indices[0]]
        axes[class_idx].imshow(sample_img)
        axes[class_idx].set_title(CLASS_NAMES[class_idx])
        axes[class_idx].axis('off')

plt.suptitle('Sample Images — One Per Class', fontsize=14)
plt.tight_layout()
plt.show()

pixel intensity

In [ ]:
# Mean pixel intensity per channel (R, G, B)
channel_names = ['Red', 'Green', 'Blue']
print("Mean pixel intensity per channel (train set, scaled 0–1):")
for i, ch in enumerate(channel_names):
    print(f"  {ch}: {X_train[:, :, :, i].mean():.4f}  |  std: {X_train[:, :, :, i].std():.4f}")
    

---

### 5) CNN Model

Code from lesson 9.1

In [ ]:
def build_tf_model(input_shape, n_labels):
    model = Sequential()

    model.add(Conv2D(filters=32, kernel_size=(3, 3), input_shape=input_shape, activation='relu'))
    model.add(MaxPool2D(pool_size=(2, 2)))

    model.add(Conv2D(filters=64, kernel_size=(3, 3), activation='relu'))
    model.add(MaxPool2D(pool_size=(2, 2)))

    model.add(Flatten())
    model.add(Dense(128, activation='relu'))
    model.add(Dropout(0.25))

    model.add(Dense(n_labels, activation='softmax'))

    model.compile(
        loss='categorical_crossentropy',
        optimizer='adam',
        metrics=['accuracy']
    )

    return model

In [ ]:
model = build_tf_model(input_shape=X_train.shape[1:], n_labels=N_LABELS)
model.summary()

---

### 6) Model Training / Fit the Model

Code from Lesson 8.2 and 9.1 was used

In [ ]:
early_stop = EarlyStopping(
    monitor='val_loss',
    mode='min',
    verbose=1,
    patience=5  # Stop if no improvement after 5 epochs
)

In [ ]:
model = build_tf_model(input_shape=X_train.shape[1:], n_labels=N_LABELS)

model.fit(
    x=X_train,
    y=y_train_cat,
    epochs=30,
    validation_data=(X_val, y_val_cat),
    verbose=1,
    callbacks=[early_stop]
)

---

Model evaluation

Code from lesson 9.1

In [ ]:
history = pd.DataFrame(model.history.history)
history.head()

Plot accuracy and loss

In [ ]:
sns.set_style("whitegrid")
history[['loss','val_loss']].plot(style='.-')
plt.title("Loss")
plt.show()

print("\n")
history[['accuracy','val_accuracy']].plot(style='.-')
plt.title("Accuracy")
plt.show()

Code from lesson 9.1, AI was used on _cat because i believe it needed a target so your model.evaluate(X_test,y_test) code didnt work

In [ ]:
model.evaluate(X_test,y_test_cat)

---

Confusion Matrix

Code from lesson 9.1

In [ ]:
def confusion_matrix_and_report(X, y, pipeline, label_map):
    """
    Print confusion matrix and report, and plot heatmap
    """

    # Predictions (convert from one-hot)
    prediction = pipeline.predict(X)
    prediction = np.argmax(prediction, axis=1)

    # True labels (convert from one-hot)
    y = np.argmax(y, axis=1)

    # Compute confusion matrix
    cm = confusion_matrix(y_true=y, y_pred=prediction)

    print('---  Confusion Matrix  ---')
    print(pd.DataFrame(
        cm,
        columns=["Actual " + sub for sub in label_map],
        index=["Predicted " + sub for sub in label_map]
    ))
    print("\n")

    print('---  Classification Report  ---')
    print(classification_report(y, prediction, target_names=label_map), "\n")

    # 🔹 Plot heatmap
    plt.figure(figsize=(6, 5))
    sns.heatmap(
        cm,
        annot=True,
        fmt='d',
        cmap='Blues',
        xticklabels=label_map,
        yticklabels=label_map
    )

    plt.xlabel('Actual')
    plt.ylabel('Predicted')
    plt.title('Confusion Matrix')
    plt.tight_layout()
    plt.show()

In [ ]:
def clf_performance(X_train,y_train,X_test,y_test,X_val, y_val,pipeline,label_map):
  """
  Print classification performance
  """
  print("#### Train Set #### \n")
  confusion_matrix_and_report(X_train,y_train,pipeline,label_map)

  print("#### Validation Set #### \n")
  confusion_matrix_and_report(X_val,y_val,pipeline,label_map)

  print("#### Test Set ####\n")
  confusion_matrix_and_report(X_test,y_test,pipeline,label_map)

In [ ]:
clf_performance(
    X_train, y_train_cat,
    X_val,   y_val_cat,
    X_test,  y_test_cat,
    model,
    label_map=CLASS_NAMES
)

---

Hyperparameter and Model Architecture Optimisation